<a href="https://colab.research.google.com/github/mmferes/PPD-2026/blob/main/Lista_de_Exerc%C3%ADcios_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lista de Exercícios 1
Exercícios propostos para ajudar a fixar o conhecimento sobre programação paralela com threads.
**Não vale nota!**

## Exercício 1 - Normalização de vetor
O código abaixo normaliza um vetor dividindo cada elemento pelo maior valor encontrado. A função *normaliza* já recebe o maior valor calculado e apenas aplica a divisão.

Paralelize essa função. Reflita: existe condição de corrida neste caso? É necessário mutex? Justifique sua resposta no comentário do código.

Requisitos:
1. Dividir o vetor entre as threads;
2. Cada thread normaliza apenas os elementos do seu intervalo; e
3. Responder nos comentários se mutex é necessário e por quê.

In [ ]:
%%writefile ex1.c

#include <stdio.h>
#include <stdlib.h>

#define N 6000000
#define N_THREADS 4

double v[N];
double maior;

void init() {
    srand(42);
    maior = 0.0;
    for (int i = 0; i < N; i++) {
        v[i] = rand() % 100000;
        if (v[i] > maior)
            maior = v[i];
    }
}

void normaliza() {
    for (int i = 0; i < N; i++)
        v[i] = v[i] / maior;
}

int main() {
    init();
    normaliza();
    printf("v[0]     = %.6f\n", v[0]);
    printf("v[N-1]   = %.6f\n", v[N-1]);
    if (v[N-1] == 1.0) {
        printf("maior  = %.6f\n", 1.0);
    } else {
        printf("maior  = %.6f\n", maior);
    }
    return 0;
}

Writing ex1.c


In [ ]:
!gcc -o ex1 ex1.c -lpthread; time ./ex1

v[0]     = 0.761668
v[N-1]   = 0.755588
maior    = 99999.000000

real	0m0.237s
user	0m0.174s
sys	0m0.053s


##Exercício 2 - Produto escalar
O produto escalar de dois vetores é a soma dos produtos elemento a elemento: resultado = Σ a[i] * b[i]. Paralelize a função *produto_escalar* dividindo o trabalho entre as threads. Cada thread deve calcular a contribuição parcial do seu intervalo e ao final combinar os resultados.

Requisitos:

1. Cada thread calcula um produto parcial localmente;
2. Usar mutex para acumular no resultado global; e
3. Comparar o resultado com a versão sequencial para validar.

In [ ]:
%%writefile ex2.c
#include <stdio.h>

#define N         9000000
#define N_THREADS 4

double a[N];
double b[N];
double resultado = 0.0;

void init() {
    for (int i = 0; i < N; i++) {
        a[i] = i * 0.3;
        b[i] = i * 0.7;
    }
}

double produto_escalar() {
    double soma = 0.0;
    for (int i = 0; i < N; i++)
        soma += a[i] * b[i];
    return soma;
}

int main() {
    init();
    resultado = produto_escalar();
    printf("Produto escalar = %.2f\n", resultado);
    return 0;
}

Writing ex2.c


In [ ]:
!gcc -o ex2 ex2.c -lpthread; time ./ex2

Produto escalar = 51029991494997229568.00

real	0m0.152s
user	0m0.069s
sys	0m0.083s


##Exercício 3 - Multiplicação matriz-vetor
A multiplicação de uma matriz M por um vetor x produz um vetor y, onde cada elemento y[i] é o produto escalar da linha i de M com x.

Paralelize a função *multiplica* distribuindo as linhas da matriz entre as threads. Analise cuidadosamente: quais variáveis são compartilhadas? Há condição de corrida? Mutex é necessário?

Requisitos:

1. Distribuir as linhas de M entre as threads
2. Cada thread calcula os elementos y[i] do seu intervalo
3. Justificar nos comentários a decisão sobre uso ou não de mutex
4. Medir e comparar o tempo com time no terminal

In [ ]:
%%writefile ex3.c
#include <stdio.h>

#define LINHAS    800
#define COLUNAS   800
#define N_THREADS 4

double M[LINHAS][COLUNAS];
double x[COLUNAS];
double y[LINHAS];

void init() {
    for (int i = 0; i < LINHAS; i++) {
        x[i] = i * 0.1;
        for (int j = 0; j < COLUNAS; j++)
            M[i][j] = i + j * 0.5;
    }
}

void multiplica() {
    for (int i = 0; i < LINHAS; i++) {
        y[i] = 0.0;
        for (int j = 0; j < COLUNAS; j++)
            y[i] += M[i][j] * x[j];
    }
}

int main() {
    init();
    multiplica();
    printf("y[0]        = %.4f\n", y[0]);
    printf("y[LINHAS-1] = %.4f\n", y[LINHAS-1]);
    return 0;
}

Writing ex3.c


In [ ]:
!gcc -o ex3 ex3.c -lpthread ; time ./ex3

y[0]        = 8517340.0000
y[LINHAS-1] = 34053380.0000

real	0m0.006s
user	0m0.002s
sys	0m0.004s


##Exercício 4 - Elevação ao quadrado
O código abaixo percorre um vetor e substitui cada elemento pelo seu quadrado.

Paralelize a função *eleva_quadrado* dividindo o vetor entre as threads. Analise com atenção: existe condição de corrida neste caso? É necessário mutex? Justifique sua resposta nos comentários do código.

Requisitos:
1. Dividir o vetor igualmente entre as threads
2. Cada thread processa apenas os elementos do seu intervalo
3. Responder nos comentários se mutex é necessário e por quê

In [ ]:
%%writefile ex4.c
#include <stdio.h>

#define N         8000000
#define N_THREADS 4

double v[N];

void init() {
    for (int i = 0; i < N; i++)
        v[i] = i * 0.5;
}

void eleva_quadrado() {
    for (int i = 0; i < N; i++)
        v[i] = v[i] * v[i];
}

int main() {
    init();
    eleva_quadrado();
    printf("v[0]   = %.2f\n", v[0]);
    printf("v[1]   = %.2f\n", v[1]);
    printf("v[N-1] = %.2f\n", v[N-1]);
    return 0;
}

Writing ex4.c


In [ ]:
!gcc -o ex4 ex4.c -lpthread; time ./ex4

v[0]   = 0.00
v[1]   = 0.25
v[N-1] = 15999996000000.25

real	0m0.094s
user	0m0.049s
sys	0m0.044s


##Exercício 5 - Contagem de palavras
O código abaixo lê um texto armazenado em um array de caracteres e conta o número total de palavras, considerando que palavras são separadas por espaço, tabulação ou quebra de linha.

Paralelize a função conta_palavras dividindo o texto em blocos entre as threads. Este exercício exige **cuidado especial nas bordas entre blocos **— uma palavra pode estar cortada exatamente no ponto de divisão entre dois intervalos.

Requisitos:

1. Dividir o texto em blocos de caracteres entre as threads
2. Cada thread conta as palavras do seu bloco localmente
3. Tratar a borda entre blocos: se o último caractere do bloco não for separador e o primeiro do próximo também não for, a palavra foi contada duas vezes — a thread deve descontar um
4. Usar mutex apenas para somar a contagem parcial ao total global


In [ ]:
%%writefile ex6.c
#include <stdio.h>
#include <string.h>
#include <ctype.h>
#include <stdlib.h>

#define TAM_TEXTO  10000000
#define N_THREADS  4

char texto[TAM_TEXTO];
int  total_palavras = 0;

void init() {
    /*
     * Gera um texto artificial com palavras de tamanho variado
     * separadas por espaços, tabulações e quebras de linha.
     */
    const char *vocab[] = {"rede", "thread", "processo", "mutex",
                           "semaforo", "kernel", "socket", "buffer"};
    const char  seps[]  = "   \t\n";
    int pos = 0;
    srand(42);
    while (pos < TAM_TEXTO - 20) {
        const char *palavra = vocab[rand() % 8];
        int len = strlen(palavra);
        memcpy(texto + pos, palavra, len);
        pos += len;
        texto[pos++] = seps[rand() % 5];
    }
    texto[TAM_TEXTO - 1] = '\0';
}

int conta_palavras() {
    int count   = 0;
    int em_palavra = 0;
    for (int i = 0; texto[i] != '\0'; i++) {
        if (!isspace((unsigned char)texto[i])) {
            if (!em_palavra) {
                count++;
                em_palavra = 1;
            }
        } else {
            em_palavra = 0;
        }
    }
    return count;
}

int main() {
    init();
    total_palavras = conta_palavras();
    printf("Total de palavras = %d\n", total_palavras);
    return 0;
}

Writing ex6.c


In [ ]:
!gcc -o ex6 ex6.c -lpthread; time ./ex6

Total de palavras = 1403551

real	0m0.116s
user	0m0.110s
sys	0m0.005s
